In [1]:
import gymnasium as gym
import gymnasium_jsbsim  # noqa: F401
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from gymnasium.envs.registration import register
from gymnasium_jsbsim.aircraft import cessna172P
from gymnasium_jsbsim.tasks import Shaping

from functools import partial
from waypoint import WaypointTask, WaypointAssessor, MinMaxObservationWrapper, WaypointVisualiser
from gymnasium_jsbsim.aircraft import cessna172P

In [2]:
my_waypoints = [(51.4, -2.3273, 5000)]
def WaypointTaskFactory(*args, **kwargs):
    """
    This function matches the library's call signature EXACTLY.
    It then injects your custom objects into your WaypointTask.
    """
    # Define your route data here (or pass it from an outer scope)
    my_assessor = WaypointAssessor(hit_bonus=100.0)
    my_plane = cessna172P

    # We return an instance of your task, explicitly mapping everything
    return WaypointTask(
        my_plane,
        my_assessor,
        my_waypoints,
    )

# register into the global Gymnasium registry
register(
    id="JSBSim-WaypointTask-Cessna172P-Shaping.STANDARD-FG-v0",
    # Since you want FlightGear, use the FG entry point from the library
    entry_point="gymnasium_jsbsim.environment:JsbSimEnv", 
    kwargs={
        "task_type": WaypointTaskFactory,     # Pass your custom class here!
        "aircraft": cessna172P,        # Pass the aircraft object
        "shaping": Shaping.STANDARD    # Pass the shaping enum
    }
)

# noFG version for faster training
register(
    id="JSBSim-WaypointTask-Cessna172P-Shaping.STANDARD-NoFG-v0",
    entry_point="gymnasium_jsbsim.environment:NoFGJsbSimEnv", 
    kwargs={
        "task_type": WaypointTaskFactory,
        "aircraft": cessna172P,
        "shaping": Shaping.STANDARD
    }
)

In [3]:
env = MinMaxObservationWrapper(gym.make("JSBSim-WaypointTask-Cessna172P-Shaping.STANDARD-NoFG-v0"))
model = PPO("MlpPolicy", env, tensorboard_log="../logs/ppo", learning_rate=1e-6, device='cpu')
model.learn(100_000, log_interval=10)
model.save(f"../models/ppo_100_000")



     JSBSim Flight Dynamics Model v1.3.0 [GitHub build 1742/commit 01ea7816288dce9e9f6a1a6dc048014bc65d699e] Apr  9 2026 10:02:36
            [JSBSim-ML v2.0]

JSBSim startup beginning ...



KeyboardInterrupt: 

In [3]:
# visualise
base_env = MinMaxObservationWrapper(gym.make("JSBSim-WaypointTask-Cessna172P-Shaping.STANDARD-FG-v0"))
model = PPO.load("../models/ppo_100_000", env=base_env)
vec_env = model.get_env()
single_env = vec_env.envs[0]
obs, info = single_env.reset()

raw_env = single_env.unwrapped

for episode in range(100000):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    done = terminated or truncated

    if episode == 1:
        vis = WaypointVisualiser(raw_env.sim)
        vis.draw_waypoints(my_waypoints)
    # single_env.render()
    # show your own rendering logic
    if done:
        obs, info = single_env.reset()

base_env.close()

/home/ondrej/Desktop/knn/.venv/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(




     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Error: timed out
Error: timed out
Error: timed out
Error: timed out
Error: timed out
Error: timed out
Error: timed out
Error: timed out


Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending: Connection refused
Socket error in Send - UDP data sending:

KeyboardInterrupt: 